In [ ]:
!pip install healpy
!pip install queryparser-python3


In [ ]:
#download data

!wget --no-check-certificate -P data/ \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_000000-003111.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_003112-005263.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_005264-006601.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_006602-007952.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_007953-010234.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_010235-012597.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_012598-014045.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_014046-015369.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_015370-016240.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_016241-017018.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_017019-017658.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_017659-018028.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_018029-018472.csv.gz" \
  "https://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/GaiaSource_018473-019161.csv.gz" \


In [ ]:
import math
import healpy as hp
from antlr4 import ParserRuleContext
from queryparser.adql import ADQLQueryTranslator

class QueryOptimizer():

    def __init__(self):
        self.ranges = [0,3112,5264,6602,7953,10235,12598, 14046, 15370, 16241]
    def _pixel_range_to_url(self,start: int, end: int, data_dir: str = "data") -> str:
        return f"{data_dir}/GaiaSource_{start:06d}-{end:06d}.csv.gz"
    
    def _pixels_to_urls(self, pixels) -> list[str]:
        urls = []
        for p in pixels:

            for i in range(len(self.ranges)):
                if i == (len(self.ranges) -1 ) and p > self.ranges[i]:
                    print (f'pixel out of range of current partitions: {p}')
                    return []
                if p >= self.ranges[i] and p <  self.ranges[i+1]:
                    start = self.ranges[i]
                    end = self.ranges[i + 1] - 1
                    url = self._pixel_range_to_url(start,end)             
                    if url not in urls:
                        urls.append(url)
                        print(f'url: {url}')
        return urls
    
    def _find_node(self, tree, rule_name, parser):
        
        if isinstance(tree, ParserRuleContext):
            if parser.ruleNames[tree.getRuleIndex()] == rule_name:
                return tree
        for i in range(tree.getChildCount()):
            result = self._find_node(tree.getChild(i), rule_name, parser)
            if result:
                return result
        return None
    
    def _find_all_nodes(self, tree, rule_name, parser, results):
        
        if isinstance(tree, ParserRuleContext):
            if parser.ruleNames[tree.getRuleIndex()] == rule_name:
                results.append(tree)
        for i in range(tree.getChildCount()):
            self._find_all_nodes(tree.getChild(i), rule_name, parser, results)
    
    def _get_leaf_value(self, tree):
        
        if tree.getChildCount() == 0:
            return tree.getText()
        for i in range(tree.getChildCount()):
            result = self._get_leaf_value(tree.getChild(i))
            if result:
                return result
            
    def _extract_circle(self, adt: ADQLQueryTranslator):
        
        circle = self._find_node(adt.tree, 'circle', adt.parser)
        if circle is None:
            return None
        ra = float(self._get_leaf_value(self._find_node(circle, 'coordinate1', adt.parser)))
        dec = float(self._get_leaf_value(self._find_node(circle, 'coordinate2', adt.parser)))
        radius = float(self._get_leaf_value(self._find_node(circle, 'radius', adt.parser)))
        return ra, dec, radius
    
    def _extract_point(self, adt: ADQLQueryTranslator):
        points = []
        self._find_all_nodes(adt.tree, 'point', adt.parser, points)
        for point in points:
            coord1 = self._find_node(point, 'coordinate1', adt.parser)
            coord2 = self._find_node(point, 'coordinate2', adt.parser)
            val1 = self._get_leaf_value(coord1)
            val2 = self._get_leaf_value(coord2)
            try:
                return float(val1), float(val2)
            except ValueError:
                continue
        return None
    
    def __call__(self, adt: ADQLQueryTranslator) -> list[str]:
        circle = self._extract_circle(adt)
        if circle is not None:
            ra, dec, radius = circle
        else:
            point = self._extract_point(adt)
            if point is not None:
                ra, dec = point
                radius = 5.0
            else:
                # no spatial constraint, caller must decide which partitions to load
                return None

        vec = hp.ang2vec(ra, dec, lonlat=True)
        pixels = hp.query_disc(nside=64, vec=vec, radius=math.radians(radius), inclusive=True)
        return self._pixels_to_urls(pixels)


In [ ]:
from pyspark.sql import SparkSession
from queryparser.adql import ADQLQueryTranslator
from pyspark.sql.functions import col, expr
from pyspark.sql.types import StringType
from utils.regex import fix_distance, fix_contains
from functools import reduce

class SparkEngine():
    def __init__(self, session: SparkSession):
        self.session = session
        self.partition_cache = {}  

    def _load_partitions(self, urls: list[str]):
        for url in urls:
            if url not in self.partition_cache:
                df = self.session.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .option("comment", "#") \
                    .option("nullValue", "") \
                    .csv(url)

                exprs = [
                    expr(f"TRY_CAST(`{f.name}` AS DOUBLE)").alias(f.name) if isinstance(f.dataType, StringType) else col(f.name)
                    for f in df.schema.fields
                ]
                df = df.select(*exprs)
                df.cache()
                self.partition_cache[url] = df

    def search(self, adt: ADQLQueryTranslator, urls: list[str]):
        
        try:
            self._load_partitions(urls)

            relevant = [self.partition_cache[url] for url in urls]
            union_df = reduce(lambda a, b: a.union(b), relevant)
            union_df.createOrReplaceTempView("gaia_source")

            sql_query = adt.to_postgresql()
            sql_query = sql_query.replace('gaiadr3.gaia_source', 'gaia_source')
            sql_query = fix_distance(sql_query)
            sql_query = fix_contains(sql_query)
            self.session.sql(sql_query).show()
        except Exception as e: 
            print(f"Spark query failed: {e}")

In [ ]:
from pyspark.sql import SparkSession
from confluent_kafka import Consumer
from typing import Dict
from optimizer.QueryOptimizer import QueryOptimizer
from queryparser.adql import ADQLQueryTranslator
from queryparser.adql import ADQLQueryTranslator
from SparkEngine.engine import SparkEngine


class AsteroideEngine():
    
    def __init__(self, kafka_config: Dict[str,str], topic: str, session: SparkSession):
        
        self.consumer = Consumer(kafka_config)
        self.consumer.subscribe([topic])
        self.optimizer = QueryOptimizer()
        self.engine = SparkEngine(session)
        
    

    def process(self, raw_query: str):

        adt = ADQLQueryTranslator()

        adt.set_query(raw_query)
        adt.parse()
        urls = self.optimizer(adt)
        self.engine.search(adt,urls)

    
    def run(self):

        while True:
            message = self.consumer.poll(timeout=1.0)
        
            if message is None:
                continue
            if message.error():
                print(f"Consumer error: {message.error()}")
                continue
            try:
                self.process(message.value().decode('utf-8'))
            except Exception as e:
                print(f"Failed to process query: {e}")
                continue


        



In [ ]:
from pyspark.sql import SparkSession
from AsteroideEngine.engine import AsteroideEngine





spark = SparkSession.builder.appName('Asteroide').getOrCreate()
kafka_config = {
    'bootstrap.servers':'localhost:9092',
    "group.id": "astro-query-group",
    "auto.offset.reset": "earliest",
    "max.poll.interval.ms": "3600000"
    }
engine = AsteroideEngine(kafka_config=kafka_config,topic='adql-queries',session=spark)
engine.run()